In [1]:
# Load packages and required files
import folium
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import os
from config import *
 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# Sites that are in our dataset
print(main_sites)
print(f'Number of sites: {len(main_sites)}')

['STREAM-gauge-2891', 'STREAM-gauge-2886', 'STREAM-gauge-2903', 'STREAM-gauge-2962', 'STREAM-gauge-2963', 'STREAM-gauge-3092', 'STREAM-gauge-3096', 'STREAM-gauge-3097', 'STREAM-gauge-3077', 'STREAM-gauge-3089', 'STREAM-gauge-3100', 'STREAM-gauge-3108', 'STREAM-gauge-3109', 'STREAM-gauge-308', 'STREAM-gauge-2203', 'STREAM-gauge-4472', 'STREAM-gauge-4431', 'STREAM-gauge-4440', 'STREAM-gauge-4442', 'STREAM-gauge-4465', 'STREAM-gauge-2804', 'STREAM-gauge-2816', 'STREAM-gauge-3776', 'STREAM-gauge-3809', 'STREAM-gauge-NA1', 'STREAM-gauge-NA2', 'STREAM-gauge-692', 'STREAM-gauge-695']
Number of sites: 28


In [3]:
# Read in data and filter for just oru study sites
metadata = pd.read_csv(metadata_filepath)
metadata = metadata[metadata.STREAM_ID.isin(main_sites)]
metadata.head(2)

,STREAM_ID,sourceID,source,site name,latitude_wgs84,longitude_wgs84,drainagearea_sqkm,state_name,time_zone,WQ_parameters
112,STREAM-gauge-3809,3216070.0,USGS,"OHIO RIVER AT IRONTON, OH",38.532056,-82.685944,157536.14180,Ohio,Eastern,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU,NO3_mgNL,C..."
140,STREAM-gauge-4431,3275500.0,USGS,"EAST FORK WHITEWATER RIVER AT RICHMOND, IN",39.806701,-84.907149,313.38879,Indiana,Eastern,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU,NO3_mgNL,P..."


In [5]:
# Sorting to largest watersheds are plotted first
metadata = metadata.sort_values('drainagearea_sqkm', ascending=False)

# Calculate the mean latitude and longitude to center the map
mean_latitude = metadata['latitude_wgs84'].mean()
mean_longitude = metadata['longitude_wgs84'].mean()

# Create a Folium map centered at the mean coordinates
m = folium.Map(location=[mean_latitude, mean_longitude], zoom_start=4)

folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    referrer_policy="strict-origin",
).add_to(m)

# Get unique states and create a color map
unique_states = metadata['state_name'].drop_duplicates().tolist()
colors = sns.color_palette('tab10', len(unique_states)).as_hex()
state_color_map = dict(zip(unique_states, colors))

In [6]:
# Add CircleMarkers for each unique STREAM_ID, colored by state
for index, row in metadata.iterrows():
    # Isolate names
    state = row['state_name']
    stream_id = row['STREAM_ID']

    # create shapfile path
    shapefile_path = os.path.join(shapefile_filepath, f"{stream_id}.shp")
    
    color = state_color_map.get(state, '#000000') # Default to black if state not in map

    folium.CircleMarker(
        location=[row['latitude_wgs84'], row['longitude_wgs84']],
        radius=5, # Adjust size as needed
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Stream ID: {row['STREAM_ID']}<br>State: {row['state_name']}",
        tooltip=row['STREAM_ID']
    ).add_to(m)

    # Adding shapefiles ot map
    wtshd = gpd.read_file(shapefile_path)
    # Add to map with matching color
    folium.GeoJson(
        wtshd,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 2,
            'fillOpacity': 0.3
        },
        popup=f"Watershed: {stream_id}"
    ).add_to(m)            

In [7]:
# Display the map
m

In [8]:
m.save(OUTPUT_filepath+'1_analysis_files_figures/watershed_map.html')